In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain_docling import DoclingLoader
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
import os
import pickle

c:\Users\rahul\Desktop\dev\llm\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [4]:
FILE_PATH = "lintransf.pdf"

pipeline_options = PdfPipelineOptions(
    do_ocr=False,
    do_formula_enrichment=True,
    do_code_enrichment=False,
    generate_page_images=False,
    generate_picture_images=False,
)

converter = DocumentConverter(
    format_options={
        "pdf": PdfFormatOption(
            pipeline_options=pipeline_options,
            backend=PyPdfiumDocumentBackend,
        )
    }
)

loader = DoclingLoader(file_path=FILE_PATH, converter=converter)

In [5]:
CACHE_FILE = "docs_cache.pkl"

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "rb") as f:
        docs = pickle.load(f)
    print(f"Loaded {len(docs)} chunks from cache.")
else:
    docs = loader.load()
    with open(CACHE_FILE, "wb") as f:
        pickle.dump(docs, f)
    print(f"Parsed and cached {len(docs)} chunks.")

Loaded 46 chunks from cache.


In [6]:
print(f"Total chunks: {len(docs)}")
print("--- Sample chunk ---")
for i in docs:
    print(i.page_content)
    print("---")

Total chunks: 46
--- Sample chunk ---
4. Linear transformations and diagonalization
This chapter is devoted to analyzing linear transformations between abstract vector spaces. The "high point" of this analysis is the so called Rank-Nullity-Theorem in its various forms. We will then focus on the case of a linear operator (that is, the case where the source and target vector spaces coincide). Here, we will see how diagonalization can help us to understand the "action" of a linear operator.
---
4.1. Linear transformations
We know the definition of a linear transformation T : R n → R m .They are mappings that preserve sums and scalar multiples. More precisely, T(x+y)= T(x)+T(y) and T(cx)= cT(x) for all x , y ↑ R n and c ↑ R. This definitions obviously makes sense for arbitrary vector spaces:
- 4.1 Definition. Let V,W be F-vector spaces (that is, V,W are both either real or complex vector spaces). A linear transformation (also: linear map) from V to W is a map
<!-- formula-not-decoded -->
s

In [7]:
VECTOR_DB_DIR = "./chroma_db"

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

def clean_metadata(docs):
    for doc in docs:
        dl_meta = doc.metadata.get("dl_meta", {})
        headings = dl_meta.get("headings", [])
        doc.metadata = {
            "source": doc.metadata.get("source", ""),
            "heading": headings[0] if headings else "",
        }
    return docs

if os.path.exists(VECTOR_DB_DIR):
    vectorstore = Chroma(persist_directory=VECTOR_DB_DIR, embedding_function=embeddings)
    print("Loaded existing vector store.")
else:
    clean_docs = clean_metadata(docs)
    vectorstore = Chroma.from_documents(clean_docs, embeddings, persist_directory=VECTOR_DB_DIR)
    print(f"Created vector store with {vectorstore._collection.count()} chunks.")

Created vector store with 46 chunks.


In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

prompt = ChatPromptTemplate.from_template("""
You are an Engineering Mathematics professor.
Use only the retrieved context.
If equations exist, render them in LaTeX.

Explain every derivation step by step.
Mention theorem names whenever applicable.
If context is insufficient, say so.

Context:
{context}

Question: {question}
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [9]:
query = "What is a linear transformation?"

answer = chain.invoke(query)
print("Answer:\n", answer)

Answer:
 A linear transformation \( T : R^n \rightarrow R^m \) is a mapping that preserves sums and scalar multiples. Specifically, it satisfies the properties \( T(x+y) = T(x) + T(y) \) and \( T(cx) = cT(x) \) for all \( x, y \in R^n \) and \( c \in R \). This definition also applies to arbitrary vector spaces, where a linear transformation from one vector space \( V \) to another vector space \( W \) maintains these properties. If \( V = W \), the linear transformation is called a linear operator.


In [15]:
# Check the retriever accuracy

query = "What is a linear transformation?"
retrieved_docs = retriever.invoke(query)  # Get the actual documents

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n📄 DOCUMENT {i}:")
    print(f"Content length: {len(doc.page_content)} chars")
    print(f"Preview: {doc.page_content[:200]}...")
    if hasattr(doc, 'metadata'):
        print(f"Metadata: {doc.metadata}")
    print("-"*40)


📄 DOCUMENT 1:
Content length: 636 chars
Preview: 4.1. Linear transformations
We know the definition of a linear transformation T : R n → R m .They are mappings that preserve sums and scalar multiples. More precisely, T(x+y)= T(x)+T(y) and T(cx)= cT(...
Metadata: {'heading': '4.1. Linear transformations', 'source': 'lintransf.pdf'}
----------------------------------------

📄 DOCUMENT 2:
Content length: 833 chars
Preview: 4.2 Examples.
1. As before if V = R n and W = R m , and A is any real m ↓ n-matrix then TA : R n → R m given by TA(x)= Ax is a linear transformation. Recall that all linear transformations from R n to...
Metadata: {'heading': '4.2 Examples.', 'source': 'lintransf.pdf'}
----------------------------------------

📄 DOCUMENT 3:
Content length: 800 chars
Preview: 4. Linear transformations and diagonalization
Both these correspondences are linear transformations. The linear transformation T can now be analyzed by the following diagram (where A = C[T]B):
<!-- fo...
Metadata: 